# 152 — Serving online, batch y streaming

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Materia resumida

Tres patrones de serving:

- **Online**: endpoint síncrono; optimiza latencia en percentiles (p95/p99, nunca el
  promedio); costo en reposo; features en ms. Para LLMs: TTFT + streaming de tokens.
- **Batch**: job periódico que materializa resultados; optimiza costo/throughput; la
  frescura queda acotada por la periodicidad; fallos benignos (relanzar).
- **Streaming**: consumo de eventos con ventanas y estado; frescura de segundos sin
  petición; la mayor complejidad operativa (lag, garantías de entrega).

Decisión en tres preguntas: ¿quién espera? → online; ¿tolera horas de edad? → batch;
¿es un evento que reacciona solo? → streaming. Capacidad:
`throughput ≈ (réplicas × lote) / latencia_por_lote` (ley de Little).


## 🧮 Ejemplo de referencia

E-commerce: 2 M usuarios, 80 000 visitantes/día.

```text
batch diario:  2.0 M predicciones/día, solo 4 % se consumen; edad hasta 24 h
online:        ~400 000 predicciones/día; pico ≈ 140 rps → 3 réplicas (50 rps c/u)
               usa la señal de la sesión actual
```

Decisión híbrida: candidatos pesados en batch (barato por unidad, tolerante a edad) +
re-ranking online con contexto de sesión (barato de computar, imposible de precalcular).


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
result = run_lab("observability", seed=152)
show(result)


## Reflexión

1. Una página compone 20 llamadas a servicios con p99 = 100 ms cada uno: ¿por qué la latencia de la página será mala aunque «solo» el 1 % de las llamadas sea lenta, y qué técnica del serving online lo mitiga?
2. ¿Qué requisito habría que cambiar en el ejemplo del e-commerce para que streaming fuera la elección correcta en lugar del híbrido batch+online?
3. Si duplicas el tamaño de lote del batching dinámico, ¿qué pasa con el throughput y con el p95, y cómo decidirías el punto de operación?
